# AlphaTensor's 47 multiplication algorithm, by hand

**Goal.** Multiply two 4×4 matrices over the field $\mathbb F_2=\{0,1\}$ using the published 47-term decomposition, then verify the identity and compare operation counts with direct multiplication and two-level Strassen.

**Where the 47 terms come from.** The paper casts algorithm discovery as a tensor decomposition game. At each move, AlphaTensor selects vectors $u_r,v_r,w_r$ and subtracts their outer product from the remaining matrix multiplication tensor. It searches for a short path to the zero tensor. The 47 triples below are *published output of that search* (the deck embeds the authors' `factorizations_f2.npz` data), not coefficients we derive by a simple formula. This notebook explains how to **use and check** them; it does not retrain AlphaTensor.

Source: Fawzi et al., *Nature* 610 (2022), DOI [10.1038/s41586-022-05172-4](https://doi.org/10.1038/s41586-022-05172-4), especially Algorithm 1 and Extended Data Fig. 1. Data: [authors' repository](https://github.com/google-deepmind/alphatensor), commit `1949163da3bef7e3eb268a3ac015fd1c2dbfc767`, `algorithms/factorizations_f2.npz`. The slide project transposes the published output factor into row-major $C$ order before encoding the masks here. The source license is at `dist/alphatensor-LICENSE.txt` beside this notebook.

For each term, $L_r=\bigoplus_i u_{ri}a_i$, $R_r=\bigoplus_j v_{rj}b_j$, $p_r=L_rR_r$, and $c_o=\bigoplus_r w_{ro}p_r$. Here $\oplus$ is XOR and multiplication of two bits is AND. Each 16-bit hexadecimal mask encodes one vector in row-major order: bit 0 is entry (1,1), bit 15 is entry (4,4).

In [1]:
# Eq. (1): each (u, v, w) is one rank-one factor of the 4×4 multiplication tensor.
TERMS_47 = [
    (0x0004, 0x0100, 0x0013),
    (0x0064, 0x0890, 0x0018),
    (0x0054, 0x0405, 0x0014),
    (0x0044, 0x0d95, 0x0010),
    (0x0001, 0x0001, 0x0105),
    (0x0101, 0x039b, 0x0100),
    (0x0501, 0x0302, 0x0102),
    (0x0367, 0x0090, 0x0008),
    (0x3002, 0x0045, 0x1004),
    (0x6002, 0x0320, 0x1002),
    (0x2002, 0x0375, 0x1000),
    (0x3057, 0x0005, 0x0004),
    (0x6507, 0x0300, 0x0002),
    (0x1000, 0x3046, 0x5400),
    (0x9008, 0x3002, 0x7602),
    (0x8008, 0x7402, 0x7420),
    (0x0301, 0x0098, 0x0108),
    (0x9a08, 0x3000, 0xa602),
    (0x0a08, 0x3020, 0xab02),
    (0x0808, 0xb820, 0x8b20),
    (0x0020, 0x58c0, 0x8090),
    (0x00a8, 0x5040, 0xc0d4),
    (0xc008, 0x5400, 0x7064),
    (0xc058, 0x0400, 0x0064),
    (0x4828, 0x5820, 0x8020),
    (0x0400, 0x9a02, 0x0320),
    (0x0088, 0xd048, 0x84d0),
    (0x0098, 0x9008, 0x0cd8),
    (0x6a08, 0x0020, 0xa002),
    (0xc0a8, 0x5000, 0xc064),
    (0x0008, 0x1000, 0x7bdf),
    (0x0c08, 0x9800, 0x0ba8),
    (0x1110, 0x000e, 0x0400),
    (0x30a8, 0x0040, 0xc004),
    (0x8880, 0xe000, 0x8420),
    (0x0c68, 0x0800, 0x00a8),
    (0x4440, 0x0e00, 0x0020),
    (0x2220, 0x00e0, 0x8000),
    (0x0002, 0x0010, 0x1009),
    (0x0c98, 0x9000, 0x0ca8),
    (0x4000, 0x5620, 0x3020),
    (0x0010, 0x940c, 0x0450),
    (0x0398, 0x0008, 0x0c08),
    (0x1288, 0x3048, 0x8400),
    (0x0200, 0x30a8, 0x8900),
    (0x8418, 0x9402, 0x0420),
    (0x9508, 0x0002, 0x0602),
]
assert len(TERMS_47) == 47
print("Loaded", len(TERMS_47), "published factors")

Loaded 47 published factors


## 1. Implement the arithmetic

The input and output matrices are flat lists of 16 bits in row-major order. The first routine performs the textbook $c_{ij}=\bigoplus_{k=0}^{3}a_{ik}b_{kj}$ calculation. The second evaluates one published factor at a time. The loops expose every XOR and multiplication; NumPy does not perform the main work.

In [2]:
def xor_selected(mask, values):
    # Eq. (2): L = ⊕_i (u_i a_i), with u_i read from the mask.
    result = 0
    for i in range(16):
        if (mask >> i) & 1:
            result ^= values[i]
    return result


def multiply_factors(a, b, terms, show_steps=False):
    # Eq. (3): C_o = ⊕_r w_ro [(⊕_i u_ri a_i)(⊕_j v_rj b_j)].
    assert len(a) == len(b) == 16
    c = [0] * 16
    for step, (u, v, w) in enumerate(terms, start=1):
        # Eq. (2): form the two linear combinations in F2.
        left = xor_selected(u, a)
        right = xor_selected(v, b)
        # Eq. (3): one scalar multiplication in F2.
        product = left & right
        # Eq. (3): scatter this product to every output selected by w.
        for o in range(16):
            if (w >> o) & 1:
                c[o] ^= product
        if show_steps and step <= 2:
            print(f"step {step:2d}: L={left}, R={right}, p={product}, output mask={w:04x}")
    return c


def multiply_direct(a, b):
    # Eq. (4): c_ij = ⊕_{k=0}^3 a_ik b_kj, for all 16 outputs.
    c = [0] * 16
    for i in range(4):
        for j in range(4):
            for k in range(4):
                c[4 * i + j] ^= a[4 * i + k] & b[4 * k + j]
    return c


A = [1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1]
B = [1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1]
C47 = multiply_factors(A, B, TERMS_47, show_steps=True)
assert C47 == multiply_direct(A, B)
print("4×4 result:", [C47[i:i + 4] for i in range(0, 16, 4)])

step  1: L=1, R=1, p=1, output mask=0013
step  2: L=1, R=1, p=1, output mask=0018
4×4 result: [[1, 1, 1, 1], [1, 1, 0, 1], [0, 0, 1, 0], [0, 0, 1, 0]]


## 2. Check the *whole identity*, not just one example

One numeric example cannot establish an algorithm for arbitrary inputs. For output $o$ and input positions $i,j$, the target coefficient is $T_{oij}=1$ exactly when $a_i b_j$ belongs to the usual matrix product. A factor contributes $w_{ro}u_{ri}v_{rj}$. We check all $16^3=4096$ coefficients in $\mathbb F_2$. This is the same tensor identity the search must satisfy.

In [3]:
def coefficient_errors(terms):
    # Eq. (5): compare T_oij with ⊕_r w_ro u_ri v_rj over F2.
    errors = 0
    for o in range(16):
        row, col = divmod(o, 4)
        for i in range(16):
            for j in range(16):
                # Eq. (4): the standard product has a_i b_j only at matching k.
                target = int(i // 4 == row and j % 4 == col and i % 4 == j // 4)
                found = 0
                # Eq. (5): reconstruct this coefficient from the factor triples.
                for u, v, w in terms:
                    found ^= ((u >> i) & 1) & ((v >> j) & 1) & ((w >> o) & 1)
                errors += found != target
    return errors


errors = coefficient_errors(TERMS_47)
print(f"Coefficient mismatches: {errors} / 4096")
assert errors == 0

Coefficient mismatches: 0 / 4096


## 3. Build the comparison algorithms

The direct method contributes one product $a_{ik}b_{kj}$ to each output. Strassen's 2×2 rule has seven products; applying it at the outer 2×2 block level and again within each block gives $7\times7=49$ factors. Over $\mathbb F_2$, plus and minus are both XOR, so the following seven masks encode the familiar Strassen formulas. Expanding an outer and inner mask maps each pair of 2×2 positions to a position in the 4×4 matrix.

In [4]:
def direct_terms():
    # Eq. (4): one rank-one term for each (output row, output col, k).
    terms = []
    for row in range(4):
        for col in range(4):
            for k in range(4):
                terms.append((1 << (4 * row + k), 1 << (4 * k + col), 1 << (4 * row + col)))
    return terms


# Eq. (6): 7 Strassen products M1,...,M7, encoded as (left, right, output) masks.
# Positions 0,1,2,3 mean upper-left, upper-right, lower-left, lower-right.
S2 = [
    (0b1001, 0b1001, 0b1001),  # M1=(a+d)(e+h), to C11,C22
    (0b1100, 0b0001, 0b1100),  # M2=(c+d)e, to C21,C22
    (0b0001, 0b1010, 0b1010),  # M3=a(f-h), to C12,C22
    (0b1000, 0b0101, 0b0101),  # M4=d(g-e), to C11,C21
    (0b0011, 0b1000, 0b0011),  # M5=(a+b)h, to C11,C12
    (0b0101, 0b0011, 0b1000),  # M6=(c-a)(e+f), to C22
    (0b1010, 0b1100, 0b0001),  # M7=(b-d)(g+h), to C11
]


def expand_mask(outer, inner):
    # Eq. (7): tensor product of two 2×2 masks, mapped into 4×4 row-major positions.
    result = 0
    for p in range(4):
        for q in range(4):
            if ((outer >> p) & 1) and ((inner >> q) & 1):
                row = 2 * (p // 2) + q // 2
                col = 2 * (p % 2) + q % 2
                result |= 1 << (4 * row + col)
    return result


def strassen_49_terms():
    # Eq. (7): 7 outer products × 7 inner products = 49 scalar products.
    return [(expand_mask(ou, iu), expand_mask(ov, iv), expand_mask(ow, iw))
            for ou, ov, ow in S2 for iu, iv, iw in S2]


TERMS_64 = direct_terms()
TERMS_49 = strassen_49_terms()
assert len(TERMS_64) == 64 and len(TERMS_49) == 49
assert coefficient_errors(TERMS_64) == coefficient_errors(TERMS_49) == 0
assert multiply_factors(A, B, TERMS_49) == C47
print("All three decompositions pass the 4096-coefficient check.")

All three decompositions pass the 4096-coefficient check.


## 4. Experiment: when does fewer multiplications help?

Count XORs for a **simple expanded implementation**: a linear combination of $k$ inputs uses $k-1$ XORs; an output receiving $q$ contributions uses $q-1$ XORs. This count does not share subexpressions, so it is not the minimum possible for Strassen or AlphaTensor. Let one XOR cost 1 and one bit multiplication cost $m$; the toy cost is $mR+X$. Vary $m$ and compare the three methods in a table. This is an operation model, **not measured runtime**.

In [5]:
def operation_counts(terms):
    # Eq. (8): X_input = Σ_r [(|u_r|-1) + (|v_r|-1)].
    input_xor = sum((u.bit_count() - 1) + (v.bit_count() - 1) for u, v, _ in terms)
    # Eq. (8): X_output = Σ_o (number of terms reaching o - 1).
    output_xor = sum(sum((w >> o) & 1 for _, _, w in terms) - 1 for o in range(16))
    return len(terms), input_xor, output_xor


methods = {"Direct": TERMS_64, "Strassen": TERMS_49, "AlphaTensor": TERMS_47}
stats = {name: operation_counts(terms) for name, terms in methods.items()}
print("Method       mult  input XOR  output XOR  total XOR")
for name, (mult, x_in, x_out) in stats.items():
    print(f"{name:12s} {mult:4d} {x_in:10d} {x_out:11d} {x_in + x_out:10d}")
print("\nToy cost = m × multiplications + total XOR")
print("m       Direct  Strassen  AlphaTensor  lowest")
for m in (1, 5, 10, 20, 50):
    # Eq. (9): cost(m) = mR + X_input + X_output.
    costs = {name: m * mult + x_in + x_out for name, (mult, x_in, x_out) in stats.items()}
    print(f"{m:<7d} {costs['Direct']:6d} {costs['Strassen']:9d} {costs['AlphaTensor']:12d}  {min(costs, key=costs.get)}")

Method       mult  input XOR  output XOR  total XOR
Direct         64          0          48         48
Strassen       49        190         128        318
AlphaTensor    47        202         138        340

Toy cost = m × multiplications + total XOR
m       Direct  Strassen  AlphaTensor  lowest
1          112       367          387  Direct
5          368       563          575  Direct
10         688       808          810  Direct
20        1328      1298         1280  AlphaTensor
50        3248      2768         2690  AlphaTensor


## 5. Final independent check against NumPy

NumPy appears **only here** as a reference implementation. With a fixed random seed, compare all three hand implemented algorithms with `numpy.matmul(A,B) % 2` across 100 random input pairs. The earlier 4096-coefficient test is stronger mathematically; this final check also catches indexing and execution mistakes.

In [6]:
import numpy as np

# Eq. (10): independent library reference C = (A @ B) mod 2.
rng = np.random.default_rng(47)
for _ in range(100):
    a_np = rng.integers(0, 2, size=(4, 4), dtype=np.int64)
    b_np = rng.integers(0, 2, size=(4, 4), dtype=np.int64)
    a, b = a_np.ravel().tolist(), b_np.ravel().tolist()
    library_result = (a_np @ b_np) % 2
    for name, terms in methods.items():
        mine = np.array(multiply_factors(a, b, terms)).reshape(4, 4)
        assert np.allclose(mine, library_result), name
    assert np.allclose(np.array(multiply_direct(a, b)).reshape(4, 4), library_result)
print("PASS: 100 random pairs, all methods agree with NumPy.")

PASS: 100 random pairs, all methods agree with NumPy.
